# 🧩 MaskedSliceJEPA — entraînement continu & évaluation retrieval (T1 → T2)

Notebook reconstruit à partir du script de *continue-training* du modèle **`MaskedSliceJEPA`**.

## Qu'est-ce que ce modèle ?
**JEPA** (*Joint-Embedding Predictive Architecture*) appliqué aux **coupes 2D axiales** des volumes IRM.
Ici il est **cross-modal** : le *contexte* vient de la **T1** (query), la *cible* de la **T2** (gallery).
Le modèle apprend donc à projeter T1 et T2 dans un **espace d'embedding commun** où la bonne paire
est la plus proche → c'est ce qui sert au **retrieval**.

### Composants (d'après `MaskedSliceJEPA`)
| Bloc | Rôle |
|---|---|
| `patch_embed` (Conv2d `patch_size`) | découpe chaque coupe en patches → tokens de dim `token_dim` |
| `pos_embed` | encodage positionnel des patches |
| `context_encoder` (Transformer, `encoder_depth`, `heads`) | encode les patches **visibles** de la T1 |
| `predictor` (Transformer, `predictor_depth`) + `predictor_pos` + `mask_token` | prédit les représentations aux positions **masquées** |
| *target encoder* (copie **EMA**, `ema_decay`) | encode la T2 (`encode_target`) — cible stable, pas de gradient |
| `mask_ratio` | fraction de patches masqués |

### Fonction de perte (`jepa_loss`)
`loss = prediction_loss + variance_weight · variance_loss + retrieval_weight · retrieval_loss`
- **prediction_loss** : prédire les tokens cibles masqués (cœur JEPA).
- **variance_loss** : anti-collapse (style VICReg) — empêche les embeddings de s'effondrer.
- **retrieval_loss** : alignement contrastif T1 ↔ T2 (optimise directement le retrieval).

### Pipeline
1. **Preprocessing** (`MRIPairPreprocessor`) : charge les volumes, sélectionne `slices_per_axis`
   coupes le long des **3 axes** (1, 2, 3).
2. **Entraînement** : cache des paires de `dataset1/train_pairs.csv` (seul split labellisé) →
   `AugmentedJepaPairDataset` (avec augmentation) → batches `{t1, t2}`.
3. **Évaluation** : embedding d'un volume = moyenne, sur les 3 axes, de la moyenne des tokens
   (`encode_context` pour T1, `encode_target` pour T2) ; similarité **cosinus** par groupe
   `(dataset, split)` ; métriques **MRR / top-1 / top-5 / top-10** vs une pseudo-vérité-terrain.


##  `masked_slice_jepa`
Le script d'origine utilise le placeholder  pour l'ensemble d'évaluation / pseudo-GT
et ses métriques. Conformément à la demande, il est remplacé partout par le **nom exact du modèle**
(`MaskedSliceJEPA` → `masked_slice_jepa`) :


> **Pré-requis** : ce notebook importe `src/jepa.py` et `src/preprocessing.py` et charge un checkpoint
> `checkpoints/masked_slice_jepa_full.pt`. Ces fichiers doivent être présents pour l'exécuter.


## 1. Imports & chemins du projet

In [10]:
from __future__ import annotations

import csv
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Racine du projet (le notebook est dans notebooks/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

try:
    from jepa import (
        AugmentedJepaPairDataset,
        MaskedSliceJEPA,
        build_axial_slice_cache,
        jepa_loss,
    )
    from preprocessing import MRIPairPreprocessor
    print("OK : modules jepa / preprocessing importés")
except ModuleNotFoundError as exc:
    print("ATTENTION : module manquant ->", exc)
    print("Ce notebook nécessite src/jepa.py et src/preprocessing.py (supprimés du projet).")


PROJECT_ROOT = c:\Users\User\PycharmProjects\hack_paris
ATTENTION : module manquant -> No module named 'jepa'
Ce notebook nécessite src/jepa.py et src/preprocessing.py (supprimés du projet).


## 2. Configuration (remplace argparse)

In [11]:
class Config:
    data_root        = PROJECT_ROOT / "data" / "ehl-paris-medical-image-retrieval"
    checkpoint       = PROJECT_ROOT / "checkpoints" / "masked_slice_jepa_full.pt"
    out_checkpoint   = PROJECT_ROOT / "checkpoints" / "masked_slice_jepa_continue.pt"
    masked_slice_jepa = PROJECT_ROOT / "data" / "masked_slice_jepa.csv"      # pseudo-verite-terrain (classement de reference)
    sample           = PROJECT_ROOT / "data" / "sample_submission.csv"
    out_dir          = PROJECT_ROOT / "experiments" / "masked_slice_jepa_continue" / "runs" / "continue"
    epochs           = 4
    batch_size       = 32
    lr               = 2e-4
    retrieval_weight = 1.0
    variance_weight  = 0.05
    slices_per_axis  = 6

cfg = Config()
print("epochs", cfg.epochs, "| batch", cfg.batch_size, "| lr", cfg.lr)


epochs 4 | batch 32 | lr 0.0002


## 3. Lecture CSV

In [12]:
def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="") as f:
        return list(csv.DictReader(f))


## 4. Sauvegarde / chargement du modèle

In [13]:
def load_model(path: Path, device: torch.device) -> "MaskedSliceJEPA":
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = MaskedSliceJEPA(
        image_size=checkpoint["image_size"],
        patch_size=checkpoint["patch_size"],
        token_dim=checkpoint["token_dim"],
        encoder_depth=checkpoint["encoder_depth"],
        predictor_depth=checkpoint["predictor_depth"],
        heads=checkpoint["heads"],
        mask_ratio=checkpoint["mask_ratio"],
        ema_decay=checkpoint.get("ema_decay", 0.996),
    )
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(device)


def save_model(model: "MaskedSliceJEPA", path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "image_size": model.image_size,
            "patch_size": model.patch_embed.proj.kernel_size[0],
            "token_dim": model.pos_embed.shape[-1],
            "encoder_depth": len(model.context_encoder.layers),
            "predictor_depth": len(model.predictor.layers),
            "heads": model.context_encoder.layers[0].self_attn.num_heads,
            "mask_ratio": model.mask_ratio,
            "ema_decay": model.ema_decay,
        },
        path,
    )


## 5. Extraction des coupes & cache d'évaluation

In [14]:
def extract_axis_slices(volume: torch.Tensor, axis: int, preprocessor: "MRIPairPreprocessor") -> torch.Tensor:
    moved = torch.movedim(volume, axis - 1, 0)
    indices = preprocessor.select_slice_indices(moved.shape[0])
    return moved[indices].float().contiguous()


def load_image_entry(
    data_root: Path,
    image_path: str,
    modality: str,
    preprocessor: "MRIPairPreprocessor",
    axes: tuple[int, ...],
) -> dict[str, object]:
    path = MRIPairPreprocessor.resolve_path(data_root, image_path)
    volume, _ = preprocessor.load_volume(path)
    return {modality: {axis: extract_axis_slices(volume, axis, preprocessor) for axis in axes}}


def build_eval_cache(
    data_root: Path,
    preprocessor: "MRIPairPreprocessor",
    axes: tuple[int, ...],
) -> tuple[dict[str, dict[str, object]], dict[str, dict[str, object]], dict[str, tuple[str, str]]]:
    query_entries: dict[str, dict[str, object]] = {}
    target_entries: dict[str, dict[str, object]] = {}
    groups: dict[str, tuple[str, str]] = {}
    for dataset in ("dataset1", "dataset2", "dataset3"):
        for split in ("val", "test"):
            qpath = data_root / dataset / f"{split}_queries.csv"
            gpath = data_root / dataset / f"{split}_gallery.csv"
            if not qpath.exists() or not gpath.exists():
                continue
            for row in read_csv(qpath):
                qid = row["query_id"]
                groups[qid] = (dataset, split)
                if qid not in query_entries:
                    query_entries[qid] = load_image_entry(data_root, row["query_image"], "t1", preprocessor, axes)
            for row in read_csv(gpath):
                tid = row["target_id"]
                if tid not in target_entries:
                    target_entries[tid] = load_image_entry(data_root, row["target_image"], "t2", preprocessor, axes)
    return query_entries, target_entries, groups


## 6. Embedding d'un volume (moyenne des tokens sur les 3 axes)

In [15]:
@torch.no_grad()
def volume_embedding(entry: dict[str, object], modality: str, model: "MaskedSliceJEPA") -> torch.Tensor:
    device = next(model.parameters()).device
    per_axis = []
    for slices in entry[modality].values():
        resized = F.interpolate(
            slices.unsqueeze(1),
            size=(model.image_size, model.image_size),
            mode="bilinear",
            align_corners=False,
        ).to(device)
        tokens = model.encode_context(resized) if modality == "t1" else model.encode_target(resized)
        per_axis.append(tokens.mean(dim=(0, 1)))
    return torch.stack(per_axis).mean(dim=0).detach().cpu()


## 7. Pseudo-vérité-terrain & galeries par groupe  *(ex-`caca`)*

In [16]:
def masked_slice_jepa_ground_truth(path: Path) -> dict[str, str]:
    df = pd.read_csv(path)
    return {
        str(row.query_id): str(row.target_id_ranking).split()[0]
        for row in df.itertuples(index=False)
    }


def gallery_by_group(data_root: Path) -> dict[tuple[str, str], list[str]]:
    result: dict[tuple[str, str], list[str]] = {}
    for dataset in ("dataset1", "dataset2", "dataset3"):
        for split in ("val", "test"):
            gpath = data_root / dataset / f"{split}_gallery.csv"
            if gpath.exists():
                result[(dataset, split)] = pd.read_csv(gpath)["target_id"].astype(str).tolist()
    return result


## 8. Évaluation retrieval (MRR / top-k)  *(ex-`evaluate_caca`)*

In [17]:
@torch.no_grad()
def evaluate_masked_slice_jepa(
    model: "MaskedSliceJEPA",
    query_entries: dict[str, dict[str, object]],
    target_entries: dict[str, dict[str, object]],
    query_groups: dict[str, tuple[str, str]],
    group_gallery: dict[tuple[str, str], list[str]],
    ground_truth: dict[str, str],
    sample: pd.DataFrame,
    out_csv: Path | None,
) -> dict[str, float]:
    model.eval()
    query_embeddings = {qid: volume_embedding(entry, "t1", model) for qid, entry in query_entries.items()}
    target_embeddings = {tid: volume_embedding(entry, "t2", model) for tid, entry in target_entries.items()}

    gallery_mats: dict[tuple[str, str], tuple[list[str], torch.Tensor]] = {}
    for group, tids in group_gallery.items():
        mat = F.normalize(torch.stack([target_embeddings[tid] for tid in tids]), dim=-1)
        gallery_mats[group] = (tids, mat)

    ranks = []
    rows = []
    for qid in sample["query_id"].astype(str):
        group = query_groups[qid]
        tids, mat = gallery_mats[group]
        query = F.normalize(query_embeddings[qid], dim=-1)
        scores = mat @ query
        order = torch.argsort(scores, descending=True).tolist()
        ranked = [tids[i] for i in order]
        rows.append({"query_id": qid, "target_id_ranking": " ".join(ranked)})
        gt = ground_truth.get(qid)
        if gt in ranked:
            ranks.append(ranked.index(gt) + 1)

    rank_tensor = torch.tensor(ranks, dtype=torch.float32)
    metrics = {
        "masked_slice_jepa_mrr": float((1.0 / rank_tensor).mean()),
        "masked_slice_jepa_top1": float((rank_tensor <= 1).float().mean()),
        "masked_slice_jepa_top5": float((rank_tensor <= 5).float().mean()),
        "masked_slice_jepa_top10": float((rank_tensor <= 10).float().mean()),
        "n_eval": float(len(ranks)),
    }
    if out_csv is not None:
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out_csv, index=False)
    model.train()
    return metrics


## 9. Device, préprocesseur & chargement du modèle

In [18]:
# Sélection du device (cuda > mps > cpu) — l'original ne gérait que mps/cpu
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

preprocessor = MRIPairPreprocessor(add_vae_reconstructions=False, slices_per_axis=cfg.slices_per_axis)
model = load_model(cfg.checkpoint, device)
print("modèle chargé :", type(model).__name__, "| image_size", model.image_size)


device: cpu


NameError: name 'MRIPairPreprocessor' is not defined

## 10. Construction des caches (entraînement + évaluation)

In [ ]:
print("building dataset1 training cache ...")
start = time.time()
train_cache = build_axial_slice_cache(
    cfg.data_root / "dataset1" / "train_pairs.csv",
    cfg.data_root,
    preprocessor,
    axes=(1, 2, 3),
)
train_ds = AugmentedJepaPairDataset(train_cache, model.image_size, preprocessor, augment=True)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
print(f"train pairs={len(train_cache)} slice-pairs={len(train_ds)} built in {time.time() - start:.1f}s")

print("building masked_slice_jepa validation cache ...")
start = time.time()
query_entries, target_entries, query_groups = build_eval_cache(cfg.data_root, preprocessor, axes=(1, 2, 3))
group_gallery = gallery_by_group(cfg.data_root)
gt = masked_slice_jepa_ground_truth(cfg.masked_slice_jepa)
sample = pd.read_csv(cfg.sample)
print(
    f"eval queries={len(query_entries)} targets={len(target_entries)} "
    f"groups={len(group_gallery)} built in {time.time() - start:.1f}s"
)


## 11. Optimiseur & évaluation de référence (epoch 0)

In [ ]:
optimizer = torch.optim.AdamW(
    [
        {"params": model.context_encoder.parameters()},
        {"params": model.predictor.parameters()},
        {"params": [model.pos_embed, model.predictor_pos, model.mask_token]},
        {"params": model.patch_embed.parameters()},
    ],
    lr=cfg.lr,
    weight_decay=1e-4,
)

best_mrr = -1.0
cfg.out_dir.mkdir(parents=True, exist_ok=True)
base_metrics = evaluate_masked_slice_jepa(
    model, query_entries, target_entries, query_groups, group_gallery, gt, sample,
    cfg.out_dir / "epoch00_submission.csv",
)
print(
    "epoch 0 "
    f"mrr={base_metrics['masked_slice_jepa_mrr']:.5f} top1={base_metrics['masked_slice_jepa_top1']:.5f} "
    f"top5={base_metrics['masked_slice_jepa_top5']:.5f} top10={base_metrics['masked_slice_jepa_top10']:.5f}"
)


## 12. Boucle d'entraînement continu

In [ ]:
history = []
for epoch in range(1, cfg.epochs + 1):
    start = time.time()
    totals = {"loss": 0.0, "prediction_loss": 0.0, "variance_loss": 0.0, "retrieval_loss": 0.0}
    count = 0
    model.train()
    for batch in train_loader:
        t1 = batch["t1"].to(device)
        t2 = batch["t2"].to(device)
        output = model(t1, t2)
        parts = jepa_loss(
            output,
            variance_weight=cfg.variance_weight,
            retrieval_weight=cfg.retrieval_weight,
        )
        optimizer.zero_grad(set_to_none=True)
        parts["loss"].backward()
        optimizer.step()
        model.update_ema()
        n = len(t1)
        for key in totals:
            totals[key] += float(parts[key].item()) * n
        count += n

    train_metrics = {key: value / max(count, 1) for key, value in totals.items()}
    metrics = evaluate_masked_slice_jepa(
        model, query_entries, target_entries, query_groups, group_gallery, gt, sample,
        cfg.out_dir / f"epoch{epoch:02d}_submission.csv",
    )
    row = {"epoch": epoch, **train_metrics, **metrics}
    history.append(row)
    pd.DataFrame(history).to_csv(cfg.out_dir / "history.csv", index=False)
    print(
        f"epoch {epoch}/{cfg.epochs} "
        f"loss={train_metrics['loss']:.5f} pred={train_metrics['prediction_loss']:.5f} "
        f"ret={train_metrics['retrieval_loss']:.5f} "
        f"mrr={metrics['masked_slice_jepa_mrr']:.5f} top1={metrics['masked_slice_jepa_top1']:.5f} "
        f"top5={metrics['masked_slice_jepa_top5']:.5f} top10={metrics['masked_slice_jepa_top10']:.5f} "
        f"time={time.time() - start:.1f}s"
    )
    if metrics["masked_slice_jepa_mrr"] > best_mrr:
        best_mrr = metrics["masked_slice_jepa_mrr"]
        save_model(model, cfg.out_checkpoint)
        print(f"saved best checkpoint: {cfg.out_checkpoint}")


---
### Notes
- **Renommage effectué** : `caca` → `masked_slice_jepa` (fonctions, métriques, argument, fichier) —
  nom exact tiré de la classe **`MaskedSliceJEPA`**.
- **Stratégie d'entraînement** : seul `dataset1/train_pairs.csv` est labellisé ; le modèle y est
  affiné, puis évalué sur val+test des 3 datasets contre une **pseudo-GT** (top-1 d'un classement de
  référence — utile car val/test n'ont pas de labels). Le meilleur checkpoint (MRR) est sauvegardé.
- **Pré-requis non présents dans le projet actuel** : `src/jepa.py`, `src/preprocessing.py`,
  `checkpoints/masked_slice_jepa_full.pt`, `data/masked_slice_jepa.csv`, `data/sample_submission.csv`.
